# 🛠️ Notebook 2: Blackjack — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/blackjack
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


We'll build `Card`, `Deck`, `Hand` with Ace handling, and a minimal `Game` loop.

In [ ]:
import random
from dataclasses import dataclass, field
from enum import Enum

class Suit(Enum):
    HEARTS = "♥"; DIAMONDS = "♦"; CLUBS = "♣"; SPADES = "♠"

RANKS = ["A","2","3","4","5","6","7","8","9","10","J","Q","K"]

@dataclass(frozen=True)
class Card:
    rank: str
    suit: Suit
    def value(self) -> int:
        if self.rank == "A":  return 11            # Ace: start at 11, downgrade later
        if self.rank in ("J","Q","K"): return 10
        return int(self.rank)
    def __repr__(self): return f"{self.rank}{self.suit.value}"


In [ ]:
@dataclass
class Deck:
    cards: list[Card] = field(default_factory=list)

    def __post_init__(self):
        if not self.cards:
            self.cards = [Card(r, s) for s in Suit for r in RANKS]

    def shuffle(self): random.shuffle(self.cards)
    def draw(self) -> Card: return self.cards.pop()


@dataclass
class Hand:
    cards: list[Card] = field(default_factory=list)
    def add(self, c: Card): self.cards.append(c)
    def value(self) -> int:
        total = sum(c.value() for c in self.cards)
        aces = sum(1 for c in self.cards if c.rank == "A")
        # Downgrade Aces from 11 → 1 while busting
        while total > 21 and aces:
            total -= 10
            aces -= 1
        return total
    def is_bust(self): return self.value() > 21
    def __repr__(self): return f"{self.cards} = {self.value()}"


In [ ]:
class Player:
    def __init__(self, name: str):
        self.name = name
        self.hand = Hand()
    def wants_hit(self) -> bool:
        return self.hand.value() < 17   # simple naive strategy

class Dealer(Player):
    def __init__(self):
        super().__init__("Dealer")
    # Standard casino rule: hit until ≥17
    def wants_hit(self) -> bool:
        return self.hand.value() < 17


class Game:
    def __init__(self, players: list[Player]):
        self.deck = Deck(); self.deck.shuffle()
        self.players = players
        self.dealer = Dealer()

    def _deal_initial(self):
        for _ in range(2):
            for p in self.players + [self.dealer]:
                p.hand.add(self.deck.draw())

    def play(self):
        self._deal_initial()
        for p in self.players:
            while p.wants_hit() and not p.hand.is_bust():
                p.hand.add(self.deck.draw())
            print(f"{p.name}: {p.hand}")
        while self.dealer.wants_hit() and not self.dealer.is_bust() if hasattr(self.dealer, "is_bust") else False:
            pass
        while self.dealer.wants_hit():
            self.dealer.hand.add(self.deck.draw())
        print(f"Dealer: {self.dealer.hand}")
        self._settle()

    def _settle(self):
        d = self.dealer.hand.value()
        for p in self.players:
            pv = p.hand.value()
            if p.hand.is_bust(): result = "BUST — dealer wins"
            elif self.dealer.hand.is_bust() or pv > d: result = "WIN"
            elif pv == d: result = "PUSH"
            else: result = "LOSE"
            print(f"  {p.name} ({pv}) vs Dealer ({d}): {result}")

random.seed(0)  # reproducible demo
Game([Player("Alice"), Player("Bob")]).play()


### Try it
- Add *betting* with a `Chips` class.
- Add *splitting pairs* (when first two cards have equal rank).
- Replace the naive `wants_hit` with a configurable `Strategy` object (Strategy Pattern).